- スケーリング前の散布図では、`New Fish`データが青色（1）の点の近くに見えるが、KNNアルゴリズムでは「青1、オレンジ4」が選択される。
（`Weight`の数値の桁がはるかに大きいため、距離計算では`Weight`の影響が非常に大きくなる）
→ <span style="color:red;">New FishデータはSmelt（0）と予測される。</span>

星印からオレンジ色の点までの距離は見た目には遠く感じられるが、KNNモデルでは数値的に、左方向の距離（`Length`）は25から15までの約10〜15の距離として測定される。

一方、青色の点までの距離（上方向の`Weight`）は180から約250までの約70〜100の距離として測定されるため、オレンジ色の点が選択される。

- スケーリング後：散布図の横軸・縦軸を同じような尺度にスケーリングすることで、2変数のスケールが近い状態になる。

<br>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd

## 1. データの準備

In [ ]:
fish = pd.read_csv(
    "https://raw.githubusercontent.com/rickiepark/hg-mldl/master/fish.csv"
)

# BreamとSmeltのみを選択
fish = fish[fish["Species"].isin(["Bream", "Smelt"])]

<br>

## 2. 入力データ（X）、ターゲットデータ（y）

In [ ]:
X_train = fish[["Length", "Weight"]]

# Bream = 1、Smelt = 0
y_train = fish["Species"].map({
    "Bream": 1,
    "Smelt": 0
})

<br>

## 3. 新しい魚を指定

In [ ]:
# Length = 25
# Weight = 150
new_fish = [[25, 150]]

<br>

## 4. スケーリング前の散布図

In [ ]:
plt.figure()

# 訓練データ - Bream
plt.scatter(
    X_train[y_train == 1]["Length"],
    X_train[y_train == 1]["Weight"],
    label="Train Bream"
)

# 訓練データ - Smelt
plt.scatter(
    X_train[y_train == 0]["Length"],
    X_train[y_train == 0]["Weight"],
    label="Train Smelt"
)

# 新しい魚
plt.scatter(
    25,
    150,
    marker="*",
    s=300,
    label="New Fish [25, 150]"
)

plt.xlabel("Length")
plt.ylabel("Weight")
plt.title("Before Scaling")
plt.legend()
plt.show()

<br>

## 5. スケーリング前のKNN学習

In [ ]:
kn = KNeighborsClassifier(n_neighbors=5)

# 元のデータで学習
kn.fit(X_train, y_train)

<br>

## 6. スケーリング前の新しい魚を予測

In [ ]:
predict = kn.predict(new_fish)

print("スケーリング前の予測:", predict)

if predict[0] == 1:
    print("予測結果: Bream")
else:
    print("予測結果: Smelt")

<br>

## 7. 標準化

In [ ]:
scaler = StandardScaler()

# 訓練データの平均と標準偏差を求めた後、標準化
# fit_transform()メソッドはfit() + transform()の役割
X_train_scaled = scaler.fit_transform(X_train)

# fit()またはfit_transform()の前にtransform()を呼び出すとError
# 新しい魚は訓練データの基準
# （すでにfit_transform()で学習した平均・標準偏差）を使って標準化
new_fish_scaled = scaler.transform(new_fish)

<br>

## 8. スケーリング後の散布図

In [ ]:
plt.figure()

# 訓練データ - Bream
plt.scatter(
    X_train_scaled[y_train == 1, 0],
    X_train_scaled[y_train == 1, 1],
    label="Train Bream"
)

# 訓練データ - Smelt
plt.scatter(
    X_train_scaled[y_train == 0, 0],
    X_train_scaled[y_train == 0, 1],
    label="Train Smelt"
)

# 新しい魚
plt.scatter(
    new_fish_scaled[0, 0],
    new_fish_scaled[0, 1],
    marker="*",
    s=300,
    label="New Fish"
)

plt.xlabel("Length (Scaled)")
plt.ylabel("Weight (Scaled)")
plt.title("After Scaling")
plt.legend()
plt.show()

<br>

## 9. スケーリング後のKNN学習

In [ ]:
kn_scaled = KNeighborsClassifier(n_neighbors=5)

# 標準化したデータで学習
kn_scaled.fit(X_train_scaled, y_train)

<br>

## 10. スケーリング後の新しい魚を予測

In [ ]:
predict_scaled = kn_scaled.predict(new_fish_scaled)

print("スケーリング後の予測:", predict_scaled)

if predict_scaled[0] == 1:
    print("予測結果: Bream")
else:
    print("予測結果: Smelt")